In [ ]:
# How prompt chaining works? 
# 1) Define the task: Start by breaking down the problem to smaller sub-tasks 
# For example --> if we want to generate a report, we can split it into steps like "gather data",
# "analyze data" and "create summary"
# 2) Create node: Each subtask becomes a node in langgraph structure. A node could be a prompt that 
# instructs a model to perform specific action such as "List key facts about X"
#3) Estabilish edges: Edges defines the sequence and dependency between nodes. For instance,output of 
# gather data node flows into analyze data node which ensures that model has necessary context to 
# proceed.
# 4) Execute langggraph: Langgraph processes the node in order passes the information along the edges.
# Model generate responses step-by-step,refining output as it progresses through the chain 
# 5) Iterate if needed: Langgraph supports conditional logic and loops,so you can revisit earlier nodes 
# or adjust the flow based on intermediate results 

 

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model = "openai/gpt-oss-20b")
result = llm.invoke("Helo")
result

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

class State(TypedDict):
    topic: str
    story: str
    improved_story: str
    final_story: str

## Nodes
def generate_story(state: State):
    msg = llm.invoke(f"Write a one sentence story premise about {state['topic']}")
    return {"story": msg.content}

def check_conflict(state: State):
    if "?" in state["story"] or "!" in state["story"]:
        return "Fail"
    return "Pass"

def improved_story(state: State):
    msg = llm.invoke(f"Enhance this story with vivid details: {state['story']}")
    return {"improved_story": msg.content}

def polish_story(state: State):
    msg = llm.invoke(f"Add an unexpected twist to this story premise: {state['improved_story']}")
    return {"final_story": msg.content}

In [ ]:
graph = StateGraph(State)
graph.add_node("generate", generate_story)
graph.add_node("improve", improved_story)
graph.add_node("polish", polish_story)

# Define edges
graph.add_edge(START, "generate")
graph.add_conditional_edges(
    "generate",
    check_conflict,
    {"Pass": "improve", "Fail": "generate"},
)
graph.add_edge("improve", "polish")
graph.add_edge("polish", END)

compiled_graph = graph.compile()

# Optional graph visualizationdisplay(Image(graph_image))
graph_image = compiled_graph.get_graph().draw_mermaid_png()
graph_image

In [ ]:
state = {"topic":"Agentic AI Systems"}
result = compiled_graph.invoke(state)
print(result)